Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Big data: the price of a pint

The LSE Growth Lab collected **3.3 million photos of menus**, posted with Google Maps reviews across 25 UK cities, and used **large language models** (AI that can read images) to pull out the prices: **12.8 million prices** from 80,000 cafes, pubs and restaurants, from 2011 to 2026. It's part of the AutoCPI project, which uses web data to measure prices in more detail than official statistics can: in June 2026, the ONS collected just 115 hospitality prices for the whole of Wales.

In this notebook we work with the prices themselves, not a summary:
1. **Big data, small slice**: read a sample, and keep memory in check.
2. **Clean**: what counts as a *pint*?
3. **Analyse**: the price of a pint over time, by place, and at chains.
4. **Map** it: by region, then pub by pub.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()     # Altair stops at 5,000 rows by default: our tables are bigger

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles

<br>

> **`disable_max_rows()`:** every row you chart is saved inside the chart, so Altair normally refuses tables of more than 5,000 rows. We turn that limit off, but still **summarise before we chart**: a chart of all 93,000 pints in this notebook would be a 4 MB file, too big for a web page.

<br><br>

---

<br>

## 1. Big data, small slice

The full dataset's main file is **249 MB**, with 12.8 million rows: too big to download in class. So we use a slice: every **beer and cider** price, 220,000 rows.

It's a **Parquet** file, a format built for big tables. It's compressed (3 MB here, against about 30 MB as a CSV), and you can read just the columns you need. `pd.read_parquet()` works like `pd.read_csv()`:

In [ ]:
beer = pd.read_parquet(DATA + "menu_beer_cider.parquet")

print(f"{len(beer):,} rows")
print(f"{beer.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")
beer.head()

<br>

In memory, the table is much bigger than the file. With big data, **read only the columns you need**:

In [ ]:
columns = ["venue_id", "vtype", "chain_type", "chain", "site", "itl1_id", "itl1", "lat", "lng", "item", "canon", "size", "price", "date"]

beer = pd.read_parquet(DATA + "menu_beer_cider.parquet", columns=columns)
beer["date"] = pd.to_datetime(beer["date"])          # real dates, for charts

print(f"{beer.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")

<br>

Each row is one price from one menu photo:
- `vtype`: venue type (`Pub & bar`, `Restaurant`, ...); `chain_type`: `independent`, `branded` chain, `managed estate` (pub companies such as Wetherspoon) or `hotel`;
- `site`: the city it was collected in; `itl1`: its region;
- `item`: as written on the menu; `canon`: the standard name (e.g. `guinness`); `size`: a size, if the menu gave one;
- `date`: when the photo was taken. Venues are anonymous (`venue_id`).

<br><br>

---

<br>

## 2. What is a pint?

Look at some of the items:

In [ ]:
beer["item"].sample(15, random_state=1)

<br>

Halves, bottles, cans, pitchers... We keep **pints in pubs** in three steps (conditions from Week 5):

In [ ]:
# 1. Pubs and bars only
in_pub = beer["vtype"] == "Pub & bar"

# 2. Not a half, a bottle, a can, a jug or a measure in ml or cl (in the item or its size)
text = beer["item"].str.lower() + " " + beer["size"].fillna("").str.lower()
not_a_pint = text.str.contains(r"half|1/2|bottle|\bcan\b|pitcher|jug|\d+\s?ml|\d+\s?cl", regex=True)

# 3. A sensible price
sensible = beer["price"].between(2, 12)

pints = beer[in_pub & ~not_a_pint & sensible].copy()      # ~ means "not"
print(f"{len(pints):,} pint prices from {pints['venue_id'].nunique():,} pubs")

<br>

> **The pattern:** `|` means "or"; `\d+\s?ml` means "digits, maybe a space, then ml", so it finds `330ml` and `500 ml`. `\bcan\b` finds the word "can", but not "American".

> **Is it perfect?** No: some menus don't say "half", and some "pints" are 500 ml. Cleaning is about being good enough, and clear about what you did. Check with `pints["item"].sample(20)`.

<br><br>

---

<br>

## 3. The price of a pint

### Over time

93,000 pints are too many to chart one by one. We **summarise** them: one row per year, with the median price and the number of prices. Read the chain from the top:
- `groupby("year")` splits the pints into one group per year;
- `["price"]` picks the column to summarise;
- `.agg(["median", "count"])` works out both, for each group;
- `reset_index()` turns `year` from the row labels (the index) back into a column, so Altair can use it.

In [ ]:
pints["year"] = pints["date"].dt.year

by_year = (pints[pints["year"].between(2017, 2026)]
           .groupby("year")["price"]
           .agg(["median", "count"])
           .reset_index())

by_year["date"] = pd.to_datetime(by_year["year"], format="%Y")     # 1 January of each year, for a time axis
by_year

In [ ]:
alt.Chart(by_year).mark_line(color="#c8102e", point={"color": "#c8102e"}).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median:Q").title("£").scale(zero=False),
    tooltip=[
        alt.Tooltip("date:T").format("%Y").title("Year"),
        alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
        alt.Tooltip("count:Q").format(",").title("Prices"),
    ],
).properties(width=450, height=250,
             title=alt.Title("The price of a pint", subtitle="Median price of a pint in UK pubs. " + SOURCE))

<br>

### By city, and at chains

For recent prices (2025 and 2026), the median in each city, keeping cities with at least 50 prices:

In [ ]:
recent = pints[pints["year"] >= 2025]

by_city = recent.groupby("site")["price"].agg(["median", "count"]).reset_index()
by_city = by_city[by_city["count"] >= 50]

alt.Chart(by_city).mark_bar(color="#c8102e").encode(
    x=alt.X("median:Q").title("Median price of a pint (£), 2025–26"),
    y=alt.Y("site:N").sort("-x").title(None),
    tooltip=[
        alt.Tooltip("site:N").title("City"),
        alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
        alt.Tooltip("count:Q").format(",").title("Prices"),
    ],
).properties(width=400)

In [ ]:
recent.groupby("chain_type")["price"].agg(["median", "count"])

<br>

Managed pub estates are much cheaper: Wetherspoon's median pint is about £3.75, against £5.50 at independents. So a city where most photos come from Wetherspoons will *look* cheap. This is **composition**: comparing different mixes of pubs, not the same pint. The research team compare like with like, and find that for main courses, the gap between the dearest and cheapest cities falls from 33 to 13 percentage points.

> **Try it:** remake the city chart with `recent[recent["chain_type"] == "independent"]`. Does the order change?

<br><br>

---

<br>

## 4. Map it

### By region

The median pint in each ITL1 region, as a choropleth (Week 7): a lookup on the region codes, with a grey layer underneath for regions with too few prices.

In [ ]:
by_region = recent.groupby("itl1_id")["price"].agg(["median", "count"]).reset_index()
by_region = by_region[by_region["count"] >= 50]

regions = alt.Data(url=MAPS + "ITL1_UK_2025.json",
                   format=alt.DataFormat(type="topojson", feature="geog"))
grey = alt.Chart(regions).mark_geoshape(fill="#e3e3e3", stroke="white")
coloured = alt.Chart(regions).transform_lookup(
    lookup="properties.id",
    from_=alt.LookupData(by_region, key="itl1_id", fields=["median", "count"]),
).mark_geoshape(stroke="white").encode(
    color=alt.Color("median:Q").title("£ a pint").scale(scheme="reds"),
    tooltip=[
        alt.Tooltip("properties.name:N").title("Region"),
        alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
        alt.Tooltip("count:Q").format(",").title("Prices"),
    ],
)

(grey + coloured).project("mercator").properties(width=320, height=420, title=alt.Title("The price of a pint by region, 2025–26", subtitle=SOURCE))

<br>

> The data covers 25 cities, so each region's figure describes its cities, not the whole region.

### Pub by pub

Zoom in on one city: each pub at its **latest** price. Sort by date, then keep the last row for each pub:

In [ ]:
latest = (recent[recent["site"] == "manchester"]
          .sort_values("date")
          .groupby("venue_id")
          .last()
          .reset_index())
print(f"{len(latest)} pubs")

councils = alt.Data(url=MAPS + "sa/msa001_greater_manchester/LAD.json",
                    format=alt.DataFormat(type="topojson", feature="geog"))
council_layer = alt.Chart(councils).mark_geoshape(fill="#f4f4f4", stroke="#999999", strokeWidth=0.5)

pubs = alt.Chart(latest).mark_circle(size=35, opacity=0.8).encode(
    longitude="lng:Q",
    latitude="lat:Q",
    color=alt.Color("price:Q").title("£ a pint").scale(scheme="reds", domain=[3, 8], clamp=True),
    tooltip=[
        alt.Tooltip("item:N").title("Item"),
        alt.Tooltip("price:Q").format(".2f").title("Price (£)"),
        alt.Tooltip("date:T").format("%b %Y").title("Photo"),
    ],
)

chart = (council_layer + pubs).project("mercator").properties(width=450, height=400,
    title=alt.Title("The latest price of a pint in each Manchester pub", subtitle=SOURCE))
chart

<br>

> `clamp=True` keeps a few very dear pints from stretching the colours: anything over £8 gets the darkest red. Manchester's collection area reaches beyond Greater Manchester, so a few pubs sit outside the council map.

Save the chart for your website:

In [ ]:
chart.save("pint_prices_manchester.json")

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. Guinness.** Use `pints[pints["canon"] == "guinness"]` for sections 3 and 4. Is a pint of Guinness dearer than the average pint?

**2. Another city.** Change `"manchester"` to another `site` (e.g. `"glasgow"`). Use the UK map `MAPS + "CTRY_UK.json"`, or a council map from map-data, as the base.

**3. A latte.** `menu_iconic_items.parquet` has every price for 12 well-known items. Repeat section 3 for a latte (`canon == "latte"`, and venue type `"Cafe & coffee"`).

**4. One city, every dish.** `cities/menu_york.parquet` has every price in York (also Wales, Glasgow, Bristol and Hull). What are York's 10 most common dishes (`canon`)?

In [ ]:
# Your code here
